# 05 - Arquivos: Excel, CSV e Parquet

Formatos de arquivo para camada Bronze/Silver/Gold.

## 1. Dados Sinteticos

In [ ]:
import pandas as pd
import numpy as np
import os, tempfile

rng = np.random.default_rng(42)
df = pd.DataFrame({
    'id': range(1, 201),
    'produto': rng.choice(['Notebook', 'Mouse', 'Teclado', 'Monitor', 'Headset'], 200),
    'valor': rng.uniform(20, 3000, 200).round(2),
    'qtd': rng.integers(1, 20, 200),
    'data': pd.date_range('2024-01-01', periods=200, freq='h'),
})
print(df.head(10))

## 2. CSV

In [ ]:
csv_path = os.path.join(tempfile.gettempdir(), 'vendas.csv')
df.to_csv(csv_path, index=False)
df_csv = pd.read_csv(csv_path, parse_dates=['data'])
print(f'Tamanho: {os.path.getsize(csv_path):,} bytes')
print(df_csv.head(3))

## 3. Excel

In [ ]:
xlsx_path = os.path.join(tempfile.gettempdir(), 'vendas.xlsx')
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Vendas', index=False)
    df.groupby('produto').agg(total=('valor', 'sum')).to_excel(writer, sheet_name='Resumo')
print(f'Tamanho: {os.path.getsize(xlsx_path):,} bytes')

## 4. Parquet

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

parquet_path = os.path.join(tempfile.gettempdir(), 'vendas.parquet')
df.to_parquet(parquet_path, engine='pyarrow', index=False)
df_pq = pd.read_parquet(parquet_path)
print(f'Tamanho: {os.path.getsize(parquet_path):,} bytes')
print(f'CSV:     {os.path.getsize(csv_path):,} bytes')
print(f'XLSX:    {os.path.getsize(xlsx_path):,} bytes')
print(f'Parquet: {os.path.getsize(parquet_path):,} bytes')

## 5. Parquet com Particoes

In [ ]:
part_dir = os.path.join(tempfile.gettempdir(), 'vendas_part')
df.to_parquet(part_dir, engine='pyarrow', index=False, partition_cols=['produto'])
import glob
arquivos = glob.glob(os.path.join(part_dir, '**/*.parquet'), recursive=True)
print(f'Arquivos gerados: {len(arquivos)}')
for a in sorted(arquivos)[:5]:
    print(f'  {a}')

## 6. Exercicio

Crie um DataFrame com 10 linhas e salve como CSV, Excel e Parquet. Compare os tamanhos dos arquivos gerados.

## Conclusao

Para datalakes, prefira Parquet (compacto, tipado, particionavel). CSV para troca com sistemas legados.